In [ ]:
import warnings
warnings.filterwarnings("ignore")
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter 

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.prompts import PromptTemplate
from langchain_community.llms import CTransformers
from langchain.chains import RetrievalQA


import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'data'
DB_FAISS_PATH = 'vectorstore/db_faiss'



# Create vector database
def create_vector_db():
    loader = DirectoryLoader(DATA_PATH,
                             glob='*.pdf',
                             loader_cls=PyPDFLoader)

    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                                   chunk_overlap=50)
    texts = text_splitter.split_documents(documents)

    # Load embeddings from the local directory
    embeddings = HuggingFaceEmbeddings(model_name='sentence_trans',  # Update with your local path
                                       model_kwargs={'device': 'cpu'})
    #model = SentenceTransformer('C:/Users/revanthteja.g/Desktop/deploy_bot_llama/sentence_trans')
    #embeddings = model.encode(sentences)
    #print(embeddings)


    db = FAISS.from_documents(texts, embeddings)
    db.save_local(DB_FAISS_PATH)


if __name__ == "__main__":
    create_vector_db()



custom_prompt_template = """Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that data is not enough, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

def set_custom_prompt():
    """
    Prompt template for QA retrieval for each vectorstore
    """
    prompt = PromptTemplate(template=custom_prompt_template,
                            input_variables=['context', 'question'])
    return prompt

#Retrieval QA Chain
def retrieval_qa_chain(llm, prompt, db):
    qa_chain = RetrievalQA.from_chain_type(llm=llm,
                                       chain_type='stuff',
                                       retriever=db.as_retriever(search_kwargs={'k': 2}),
                                       #return_source_documents=True,
                                       chain_type_kwargs={'prompt': prompt}
                                       )
    return qa_chain

#Loading the model
def load_llm():
    # Load the locally downloaded model here
    llm = CTransformers(
        #model = "TheBloke/Llama-2-7B-Chat-GGML",
        model="llama-2-7b-chat.ggmlv3.q8_0.bin",
        model_type="llama",
        #max_new_tokens = 512,
        #temperature = 0.5
        max_new_tokens = 500,
        temperature = 0.1
    )
    return llm

#QA Model Function
def qa_bot():
    embeddings = HuggingFaceEmbeddings(model_name='sentence_trans',
                                       model_kwargs={'device': 'cpu'})
    db = FAISS.load_local(DB_FAISS_PATH, embeddings,allow_dangerous_deserialization=True)
    llm = load_llm()
    qa_prompt = set_custom_prompt()
    qa = retrieval_qa_chain(llm, qa_prompt, db)

    return qa



#output function
def final_result(query):
    qa_result = qa_bot()
    response = qa_result({'query': query})
    return response
import time
import asyncio
import concurrent.futures
from flask import Flask, render_template, request, jsonify

app = Flask(__name__)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/ask', methods=['POST'])
async def ask():  # Keep the route asynchronous
    data = request.get_json()
    print(data)
    if 'query' not in data:
        return jsonify({"error": "Query parameter is required"}), 400
    query = data['query']
    start = time.time()
    
    # Use ThreadPoolExecutor for async behavior
    with concurrent.futures.ThreadPoolExecutor() as executor:
        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(executor, final_result, query)
    
    result = response['result']
    end = time.time()
    response_time = end - start
    response_time1 = f'response time - {response_time} seconds \n'
    print(response_time1)
    raw_response = response_time1 + result
    steps = raw_response.split("\n")
    formatted_response = "<br>".join(steps)
    
    return jsonify({'result': formatted_response})

if __name__ == "__main__":
    app.run(debug=False,port=5001) 

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
